# 06 · Narrator and numeric grounding

**Role of this layer.** The narrator writes a short plain-English summary. It
never sees data series, only a **facts sheet** of numbers the executor computed,
already rounded (4 significant figures). It is told to use only those numbers,
exactly as displayed, and to do no arithmetic.

`check_grounding` then verifies that deterministically: every number in the
text must be a correct rounding of a computed fact (or its percentage form), a
plan parameter, a number from the request or a warning, an annualisation
factor, or a date within the data range. Without an API key a
deterministic template writes the narrative instead, and it passes the same check.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.narrator import check_grounding
from quantlab.scenarios import SCENARIOS_BY_ID

scenario = SCENARIOS_BY_ID["B"]  # periods of high volatility
session = run_request(scenario.request, list(scenario.dataset_ids), "mock", approve=True)
facts = session.narrative.facts
pd.DataFrame([{"fact": f.fact_id, "value": f.value, "display": f.display} for f in facts]).head(20)

,fact,value,display
0,s1.n_obs,14040,"14,040"
1,s1.start,1971-02-05,1971-02-05
2,s1.end,2026-10-13,2026-10-13
3,s1.last_value,27204.38,"27,204"
4,s2.n_obs,14039,"14,039"
5,s2.start,1971-02-08,1971-02-08
6,s2.end,2026-10-13,2026-10-13
7,s2.last_value,-0.001524,-0.001524
8,s3.n_obs,14019,"14,019"
9,s3.start,1971-03-09,1971-03-09


In [3]:
narrative = session.narrative
print(f"source: {narrative.source}")
print(narrative.text)
print("\ngrounding:", narrative.grounding)

source: template
The analysis used NASDAQCOM (1971-02-05 to 2026-10-13). The latest rolling volatility (s3) is 0.1402 on 2026-10-13. 2,579 of 13,767 observations (a share of 0.1873) were flagged, in 109 distinct episodes. Approved assumptions: material assumption (G12).

grounding: grounded=True numbers_checked=8 ungrounded=[]


The check catches invented numbers, and accepts correct roundings and
percentages. (`s3.last_value` is the latest volatility in the facts sheet.)

In [4]:
vol = next(f for f in facts if f.fact_id == "s3.last_value").display
share = next(f for f in facts if f.fact_id == "s4.share_flagged").display
for sentence in [
    f"Latest volatility is {vol}, and a share of {share} of the days was flagged.",
    f"Latest volatility is about {float(vol) * 100:.1f}%.",
    "The Sharpe ratio of the strategy is 1.83.",
    "Volatility peaked in 1955.",
    "The difference is significant at the 5% level.",
]:
    result = check_grounding(sentence, facts, session.plan, [scenario.request])
    print(f"{'OK ' if result.grounded else 'BAD'} {sentence}  ungrounded={result.ungrounded}")

OK  Latest volatility is 0.1402, and a share of 0.1873 of the days was flagged.  ungrounded=[]
OK  Latest volatility is about 14.0%.  ungrounded=[]
BAD The Sharpe ratio of the strategy is 1.83.  ungrounded=['1.83']
BAD Volatility peaked in 1955.  ungrounded=['1955']
BAD The difference is significant at the 5% level.  ungrounded=['5']


With an API key, the same facts go to the LLM narrator (live mode):

In [5]:
if settings.has_api_key:
    from quantlab.narrator import narrate

    live = narrate(
        session.request,
        session.plan,
        session.execution.artifacts,
        session.warnings(),
        "live",
        settings,
    )
    print(live.source, live.model, live.error)
    print(live.text)
    print("grounding:", live.grounding)
else:
    print(
        f"No {settings.api_key_name} set: skipping the LLM narrator (the template is used above)."
    )

No OPENAI_API_KEY set: skipping the LLM narrator (the template is used above).
